# Detección de objetos con YOLO: encontrando placas de autos

Hasta ahora nuestras redes respondían a la pregunta **"¿qué hay en esta imagen?"** con una sola etiqueta para toda la foto (piedra, papel o tijeras). Con Grad-CAM vimos, de forma aproximada, **dónde** miraba la red para decidir.

La **detección de objetos** va un paso más allá: el modelo responde **qué** objetos hay y **dónde** está cada uno, dibujando una **caja delimitadora** (*bounding box*) alrededor de cada objeto.

| Tarea | Pregunta que responde | Salida |
|-------|----------------------|--------|
| Clasificación | ¿Qué hay en la imagen? | Una etiqueta por imagen |
| **Detección** | **¿Qué hay y dónde está?** | **Una caja + etiqueta + confianza por objeto** |
| Segmentación | ¿Qué píxeles pertenecen a cada objeto? | Una máscara por objeto |

En este tutorial entrenaremos un detector de **placas de autos** (*license plates*), el primer paso de cualquier sistema de lectura automática de placas: control de estacionamientos, peajes, fotopapeletas, etc.

### Plan del notebook

1. Preparar el entorno e instalar Ultralytics.
2. Entender qué es YOLO.
3. Conocer el dataset y el **formato de etiquetas YOLO**.
4. Probar un modelo **preentrenado** y ver por qué no nos sirve tal cual.
5. **Entrenar** (fine-tuning) nuestro propio detector de placas.
6. **Evaluar** el modelo: IoU, precisión, recall y mAP.
7. Analizar errores y **recortar las placas** detectadas.

## 1. Preparación del entorno

Usaremos **[Ultralytics](https://docs.ultralytics.com/)**, la librería que mantiene la familia de modelos YOLO. Con muy pocas líneas permite entrenar, evaluar y usar detectores de objetos.

El proyecto usa `uv` como gestor de paquetes. Si Ultralytics no está instalado, ejecuta en la terminal, dentro de la carpeta del proyecto:

```bash
uv add ultralytics
```

Esto lo registra en `pyproject.toml` y `uv.lock`, para que todos tengan las mismas versiones.

### Librerías, semilla, dispositivo y rutas

Tres detalles de esta celda:

- **Dispositivo:** Ultralytics usa `0` para la primera GPU NVIDIA, `"mps"` para Apple Silicon y `"cpu"` si no hay GPU.
- **`DATA_DIR`:** la carpeta del dataset. Usamos `.resolve()` para convertirla en ruta **absoluta**, porque Ultralytics la necesita así.
- **`RUNS_DIR`:** la carpeta donde se guardarán los entrenamientos (pesos, gráficos, métricas). También la hacemos absoluta: si le pasáramos una ruta relativa, Ultralytics la colocaría dentro de su propia carpeta global de configuración y sería difícil encontrarla.

In [ ]:
from pathlib import Path
import random
import time
import warnings

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image

import ultralytics
from ultralytics import YOLO

# Semilla para que los resultados sean reproducibles
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Dispositivo en el formato que espera Ultralytics
if torch.cuda.is_available():
    DEVICE = 0              # primera GPU NVIDIA
elif torch.backends.mps.is_available():
    DEVICE = "mps"          # GPU de Apple (M1/M2/M3...)
else:
    DEVICE = "cpu"

# En Apple Silicon, PyTorch avisa en cada iteración que algunas operaciones no son deterministas
warnings.filterwarnings("ignore", message=".*does not have a deterministic implementation.*")

# Rutas absolutas del dataset y de los resultados
DATA_DIR = Path("../data/license-plates").resolve()
RUNS_DIR = Path("runs").resolve()

print(f"Ultralytics {ultralytics.__version__} | PyTorch {torch.__version__} | Dispositivo: {DEVICE}")
print(f"Dataset:    {DATA_DIR}")
print(f"Resultados: {RUNS_DIR}")

Definimos también una pequeña función para mostrar imágenes, que usaremos a lo largo del notebook. Acepta una ruta, una imagen PIL o un arreglo de NumPy.

In [ ]:
def mostrar(imagen, titulo=None, ax=None, figsize=(8, 6)):
    # Muestra una imagen (ruta, PIL o arreglo RGB) con matplotlib.
    if isinstance(imagen, (str, Path)):
        imagen = Image.open(imagen)
    if ax is None:
        _, ax = plt.subplots(figsize=figsize)
    ax.imshow(imagen)
    ax.axis("off")
    if titulo:
        ax.set_title(titulo, fontsize=10)
    return ax

## 2. ¿Qué es YOLO?

**YOLO** significa *You Only Look Once* ("solo miras una vez"). Fue propuesto en 2016 por Joseph Redmon y cambió la detección de objetos por una idea simple:

- Los detectores anteriores funcionaban en **dos etapas**: primero proponían miles de regiones candidatas y luego clasificaban cada una. Eran precisos, pero lentos.
- YOLO hace todo en **una sola pasada** de la red: divide la imagen en una grilla y, para cada celda, predice directamente las cajas, las clases y la confianza. Por eso es lo bastante rápido para video en tiempo real.

Por dentro, YOLO es una **CNN** como las que ya conocemos, con tres partes:

```
imagen → [Backbone] → [Neck] → [Head] → cajas + clases + confianzas
          extrae        combina    predice
          características  varias escalas
```

- **Backbone:** capas convolucionales que extraen características (bordes → formas → objetos), igual que la ResNet del workshop.
- **Neck:** combina características de varias resoluciones, para detectar objetos grandes **y** pequeños.
- **Head:** produce las predicciones finales: coordenadas de la caja, clase y confianza.

### Versiones y tamaños

Desde 2016 han salido muchas versiones (YOLOv3, v5, v8, YOLO11...). Usaremos **YOLO26**, la más reciente de Ultralytics. Cada versión viene en varios tamaños:

| Modelo | Parámetros aprox. | Velocidad | Precisión |
|--------|-------------------|-----------|-----------|
| `yolo26n` (nano) | 2.4 M | la más rápida | menor |
| `yolo26s` (small) | 10 M | rápida | media |
| `yolo26m` (medium) | 20 M | media | alta |
| `yolo26l` / `yolo26x` | 25 a 55 M | lenta | la más alta |

Usaremos **`yolo26n`**: es suficiente para una sola clase y se entrena en minutos en una laptop.

## 3. El dataset de placas

El dataset tiene fotos de autos (principalmente de la India), cada una con **una placa anotada**. Tiene una sola clase: `license_plate`.

Está organizado en el **formato YOLO**, el estándar que espera Ultralytics:

```
license-plates/
├── images/
│   ├── train/   ← imágenes de entrenamiento (.jpg)
│   └── val/     ← imágenes de validación (.jpg)
└── labels/
    ├── train/   ← un .txt por imagen, con el MISMO nombre
    └── val/
```

La regla clave: para cada imagen `images/train/foto.jpg` existe una etiqueta `labels/train/foto.txt`. Ultralytics encuentra las etiquetas reemplazando `images` por `labels` en la ruta.

Contemos cuántos archivos hay en cada partición:

In [ ]:
SPLITS = ["train", "val"]

for split in SPLITS:
    n_imagenes = len(list((DATA_DIR / "images" / split).glob("*.jpg")))
    n_etiquetas = len(list((DATA_DIR / "labels" / split).glob("*.txt")))
    print(f"{split:>5s} → {n_imagenes} imágenes | {n_etiquetas} etiquetas")

print("\nContenido de la carpeta:")
for elemento in sorted(DATA_DIR.iterdir()):
    print("  ", elemento.name + ("/" if elemento.is_dir() else ""))

### Observaciones sobre este dataset

- Hay **1,526 imágenes de train y 169 de validación** (90% / 10%). **No hay partición de test**, así que usaremos validación para evaluar. En un proyecto real conviene reservar un test aparte, que no se use para tomar ninguna decisión.
- Muchas imágenes son **frames de videos** (archivos `videoN_XXXX.jpg`). En train conviven fotos sueltas y frames de algunos videos; la validación está formada **casi entera por frames de otros videos** (sobre todo `video5`, `video6`, `video8` y `video9`). Lo revisamos abajo.
- Las carpetas `train/` y `val/` que están en la raíz son **copias** de `labels/train` y `labels/val`. Las ignoramos.
- El archivo `dataset.yaml` que trae el dataset apunta a rutas de **Kaggle** (donde fue publicado), que no existen en nuestra computadora:

In [ ]:
print((DATA_DIR / "dataset.yaml").read_text())

Verifiquemos de dónde vienen las imágenes de cada partición: agrupamos los frames por el nombre del video.

In [ ]:
def origen(ruta):
    # 'video8_2090.jpg' → 'video8'; cualquier otra imagen → 'foto suelta'
    nombre = Path(ruta).name
    return nombre.split("_")[0] if nombre.startswith("video") else "foto suelta"


for split in SPLITS:
    conteo = pd.Series([origen(r) for r in (DATA_DIR / "images" / split).glob("*.jpg")]).value_counts()
    print(f"{split}: " + ", ".join(f"{k} ({v})" for k, v in conteo.items()))

Esto tiene dos consecuencias importantes para la evaluación:

1. **Casi no hay fuga de datos** (*data leakage*): los videos de validación prácticamente no aparecen en train (solo `video4` está en ambos, con 2 frames en val). Si los mismos videos estuvieran en las dos particiones, el modelo se evaluaría con frames casi idénticos a los que usó para aprender, y las métricas serían engañosamente altas.
2. **La validación es poco variada:** frames consecutivos de un mismo video se parecen mucho entre sí. Aunque son 169 imágenes, equivalen a **unas pocas escenas**. Las métricas que obtengamos describen bien el desempeño en ese tipo de videos, pero no garantizan el mismo resultado en fotos muy distintas.

Por eso, más adelante crearemos **nuestro propio archivo YAML** con las rutas correctas.

## 4. El formato de etiquetas YOLO

Cada línea de un `.txt` describe **un objeto** con 5 números:

```
<clase> <x_centro> <y_centro> <ancho> <alto>
```

- `clase`: el número de la clase (aquí siempre `0` = `license_plate`).
- `x_centro`, `y_centro`: el **centro** de la caja.
- `ancho`, `alto`: el tamaño de la caja.

Las cuatro coordenadas están **normalizadas entre 0 y 1**, es decir, divididas por el ancho `W` y el alto `H` de la imagen. Así la etiqueta sirve aunque la imagen se redimensione.

Para dibujar la caja necesitamos convertirla a **esquinas en píxeles** `(x1, y1, x2, y2)`:

$$x_1 = (x_c - w/2)\cdot W \qquad y_1 = (y_c - h/2)\cdot H \qquad x_2 = (x_c + w/2)\cdot W \qquad y_2 = (y_c + h/2)\cdot H$$

Veamos el contenido real de una etiqueta:

In [ ]:
imgs_train = sorted((DATA_DIR / "images" / "train").glob("*.jpg"))
imgs_val = sorted((DATA_DIR / "images" / "val").glob("*.jpg"))


def ruta_etiqueta(ruta_img):
    # images/<split>/nombre.jpg  →  labels/<split>/nombre.txt
    ruta_img = Path(ruta_img)
    return DATA_DIR / "labels" / ruta_img.parent.name / f"{ruta_img.stem}.txt"


ejemplo = imgs_train[0]
print("Imagen: ", ejemplo.name)
print("Etiqueta:", ruta_etiqueta(ejemplo).name)
print("Contenido del .txt:", ruta_etiqueta(ejemplo).read_text())
print("Tamaño de la imagen (ancho, alto):", Image.open(ejemplo).size)

Ahora escribimos tres funciones pequeñas que usaremos en todo el notebook:

- `leer_etiquetas()`: lee un `.txt` y devuelve un arreglo con una fila por objeto.
- `yolo_a_xyxy()`: aplica las fórmulas de arriba para pasar a esquinas en píxeles.
- `dibujar_cajas()`: dibuja rectángulos sobre una imagen con matplotlib.

In [ ]:
def leer_etiquetas(ruta_txt):
    # Devuelve un arreglo (N, 5): clase, xc, yc, w, h. Si no hay objetos, (0, 5).
    ruta_txt = Path(ruta_txt)
    if not ruta_txt.exists() or ruta_txt.stat().st_size == 0:
        return np.zeros((0, 5))
    return np.loadtxt(ruta_txt, ndmin=2)


def yolo_a_xyxy(etiquetas, ancho, alto):
    # Convierte (xc, yc, w, h) normalizados a (x1, y1, x2, y2) en píxeles.
    xc, yc, w, h = etiquetas[:, 1], etiquetas[:, 2], etiquetas[:, 3], etiquetas[:, 4]
    return np.stack([(xc - w / 2) * ancho, (yc - h / 2) * alto,
                     (xc + w / 2) * ancho, (yc + h / 2) * alto], axis=1)


def dibujar_cajas(ax, cajas, color, textos=None):
    # Dibuja cajas (x1, y1, x2, y2) en un eje de matplotlib, con un texto opcional encima.
    for i, (x1, y1, x2, y2) in enumerate(cajas):
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, lw=2, edgecolor=color))
        if textos is not None:
            ax.text(x1, y1 - 3, textos[i], color="white", fontsize=8,
                    bbox=dict(facecolor=color, pad=1, lw=0))


# Aplicamos las funciones a la imagen de ejemplo
etiquetas = leer_etiquetas(ruta_etiqueta(ejemplo))
ancho, alto = Image.open(ejemplo).size
cajas = yolo_a_xyxy(etiquetas, ancho, alto)
print("Etiqueta YOLO (normalizada):", etiquetas[0])
print("Caja en píxeles (x1, y1, x2, y2):", cajas[0].round(1))

ax = mostrar(ejemplo, "Caja real (ground truth) dibujada desde el .txt")
dibujar_cajas(ax, cajas, "lime", ["license_plate"])
plt.show()

A la caja anotada por una persona se le llama **ground truth** (la "verdad"). Es lo que el modelo debe aprender a predecir.

Veamos varias imágenes de entrenamiento con sus cajas:

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, ruta in zip(axes.flat, random.sample(imgs_train, 8)):
    img = Image.open(ruta)
    cajas = yolo_a_xyxy(leer_etiquetas(ruta_etiqueta(ruta)), *img.size)
    mostrar(img, f"{img.size[0]}×{img.size[1]} px", ax=ax)
    dibujar_cajas(ax, cajas, "lime")
plt.suptitle("Imágenes de entrenamiento con su placa anotada", fontsize=14)
plt.tight_layout()
plt.show()

### ¿Qué observamos?

- Las fotos son **muy variadas**: autos de frente y de atrás, de cerca y de lejos, de día y de noche, con distintos ángulos.
- Las imágenes tienen **tamaños distintos**. No es un problema: YOLO las redimensiona a 640 px y las etiquetas normalizadas se adaptan solas.
- La placa ocupa, en general, una **porción pequeña** de la imagen. Vamos a cuantificarlo.

## 5. Análisis exploratorio de las anotaciones

Antes de entrenar hay que conocer los datos. En detección interesa sobre todo:

- **Cuántos objetos** hay por imagen.
- **Qué tan grandes** son los objetos. Los objetos pequeños son los más difíciles de detectar, porque ocupan pocos píxeles.
- **Qué forma** tienen (relación ancho/alto) y **dónde** suelen aparecer.

Construimos una tabla con **todas** las anotaciones:

In [ ]:
filas = []
for split in SPLITS:
    for ruta_img in sorted((DATA_DIR / "images" / split).glob("*.jpg")):
        ancho_img, alto_img = Image.open(ruta_img).size   # PIL solo lee la cabecera: es rápido
        for _, xc, yc, w, h in leer_etiquetas(ruta_etiqueta(ruta_img)):
            filas.append({"split": split, "imagen": ruta_img.name,
                          "ancho_img": ancho_img, "alto_img": alto_img,
                          "xc": xc, "yc": yc, "w": w, "h": h})

df_anot = pd.DataFrame(filas)

# Métricas derivadas
df_anot["area_%"] = df_anot["w"] * df_anot["h"] * 100                              # % del área de la imagen
df_anot["aspecto"] = (df_anot["w"] * df_anot["ancho_img"]) / (df_anot["h"] * df_anot["alto_img"])  # ancho / alto en píxeles

objetos_por_imagen = df_anot.groupby(["split", "imagen"]).size()
print(f"Total de anotaciones: {len(df_anot)}")
print(f"Objetos por imagen: mínimo {objetos_por_imagen.min()}, máximo {objetos_por_imagen.max()}")
df_anot[["ancho_img", "alto_img", "area_%", "aspecto"]].describe().round(2)

In [ ]:
df_anot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

axes[0].hist(df_anot["area_%"].clip(upper=20), bins=40, color="steelblue")
axes[0].set(title="Tamaño de la placa (valores > 20% agrupados en 20)", xlabel="% del área de la imagen",
            ylabel="n.º de placas")

axes[1].hist(df_anot["aspecto"].clip(upper=8), bins=50, color="darkorange")
axes[1].set(title="Forma de la placa", xlabel="relación ancho / alto", ylabel="n.º de placas")

axes[2].scatter(df_anot["xc"], df_anot["yc"], s=4, alpha=0.4, color="seagreen")
axes[2].set(title="Posición del centro de la placa", xlabel="x (0 = izquierda)", ylabel="y (0 = arriba)",
            xlim=(0, 1), ylim=(1, 0))    # invertimos y: en imágenes, y crece hacia abajo
axes[2].set_aspect("equal")

plt.tight_layout()
plt.show()

### ¿Qué observamos?

- **Un objeto por imagen:** cada foto tiene exactamente una placa anotada.
- **Placas pequeñas:** la mitad de las placas ocupa **menos del 2%** del área de la imagen, y 3 de cada 4 ocupan menos del 3.6%. Solo unas pocas fotos son primeros planos de la placa. El tamaño pequeño es el principal reto del problema.
- **Forma alargada:** las placas son rectángulos horizontales; la mitad central está entre **3 y 4.2 veces más ancha que alta**. El modelo aprenderá que las cajas "típicas" tienen esa forma.
- **Posición:** los centros se concentran en el **centro horizontal** (x ≈ 0.5) y en la **mitad inferior** de la imagen (y entre 0.5 y 0.8). Tiene sentido: quien toma la foto centra el auto, y la placa va en la parte baja del auto.

Revisemos también los tamaños de imagen más frecuentes:

In [ ]:
tamanos = (df_anot["ancho_img"].astype(str) + "×" + df_anot["alto_img"].astype(str)).value_counts()
print(f"Tamaños distintos de imagen: {len(tamanos)}")
tamanos.head(8)

## 6. El archivo de configuración `data.yaml`

Ultralytics necesita un archivo YAML que le diga **dónde** están los datos y **cómo se llaman** las clases:

| Clave | Significado |
|-------|-------------|
| `path` | Carpeta raíz del dataset (ruta absoluta) |
| `train` | Carpeta de imágenes de entrenamiento, relativa a `path` |
| `val` | Carpeta de imágenes de validación, relativa a `path` |
| `names` | Diccionario `{número de clase: nombre}` |

Lo creamos desde Python con la librería `yaml` y lo guardamos dentro del dataset.

In [ ]:
CLASES = {0: "license_plate"}

config = {
    "path": str(DATA_DIR),       # raíz del dataset (ruta absoluta)
    "train": "images/train",     # rutas relativas a 'path'
    "val": "images/val",
    "names": CLASES,
}

DATA_YAML = DATA_DIR / "data.yaml"
DATA_YAML.write_text(yaml.safe_dump(config, sort_keys=False))
print(DATA_YAML.read_text())

## 7. Primera prueba: un modelo YOLO preentrenado

Antes de entrenar, probemos un modelo YOLO26n **ya entrenado** con **COCO**, un dataset público de 80 clases de objetos cotidianos (personas, autos, perros, sillas...).

`YOLO("yolo26n.pt")` descarga los pesos la primera vez y los guarda en la carpeta actual.

In [ ]:
modelo_coco = YOLO("yolo26n.pt")

n_params = sum(p.numel() for p in modelo_coco.model.parameters())
print(f"Parámetros: {n_params / 1e6:.1f} millones")
print(f"Clases que conoce ({len(modelo_coco.names)}):")
print(", ".join(modelo_coco.names.values()))

Revisa la lista: aparecen `car`, `truck`, `bus`, `motorcycle`... pero **no existe la clase "placa"**. Un modelo solo puede detectar las clases con las que fue entrenado.

Hagamos predicciones sobre algunas imágenes de nuestro dataset con `modelo.predict()`:

- `conf=0.25`: solo se aceptan detecciones con confianza de al menos 25%.
- Devuelve una **lista de resultados**, uno por imagen.
- `resultado.plot()` devuelve la imagen con las cajas dibujadas. Viene en formato **BGR** (la convención de OpenCV), así que invertimos los canales con `[..., ::-1]` para verla en **RGB** con matplotlib.

In [ ]:
muestra_val = random.sample(imgs_val, 4)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, ruta in zip(axes, muestra_val):
    resultado = modelo_coco.predict(ruta, conf=0.25, device=DEVICE, verbose=False)[0]
    detectado = sorted({resultado.names[int(c)] for c in resultado.boxes.cls}) or ["nada"]
    mostrar(resultado.plot()[..., ::-1], "Detecta: " + ", ".join(detectado), ax=ax)
plt.suptitle("Modelo preentrenado en COCO (sin entrenar con placas)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(20, 5))

ruta = "/Users/marcell/Documents/codigo_by_tecsup/datascience-g8/data/zoologico.jpeg"
resultado = modelo_coco.predict(ruta, conf=0.5, device=DEVICE, verbose=False)[0]
detectado = sorted({resultado.names[int(c)] for c in resultado.boxes.cls}) or ["nada"]
mostrar(resultado.plot()[..., ::-1], "Detecta: " + ", ".join(detectado), ax=axes)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(20, 5))

ruta = "/Users/marcell/Documents/codigo_by_tecsup/datascience-g8/data/frutas.jpeg"
resultado = modelo_coco.predict(ruta, conf=0.25, device=DEVICE, verbose=False)[0]
detectado = sorted({resultado.names[int(c)] for c in resultado.boxes.cls}) or ["nada"]
mostrar(resultado.plot()[..., ::-1], "Detecta: " + ", ".join(detectado), ax=axes)
plt.show()

El modelo de COCO detecta **autos** sin problema, pero no sabe lo que es una placa.

La buena noticia: este modelo ya aprendió a "ver" bordes, texturas, formas y objetos con miles de imágenes. En lugar de empezar desde cero, lo usaremos como punto de partida y lo **ajustaremos** a nuestra nueva clase. Es la misma idea del **transfer learning** que usamos con ResNet-18 en el workshop, aplicada a detección.

### ¿Qué contiene un resultado?

Antes de entrenar, veamos cómo se leen las predicciones desde el código. `resultado.boxes` contiene las cajas en varios formatos:

| Atributo | Formato |
|----------|---------|
| `xyxy` | esquinas `(x1, y1, x2, y2)` en píxeles |
| `xywhn` | centro + ancho y alto, **normalizados** (igual que el `.txt` de YOLO) |
| `conf` | confianza de cada caja (0 a 1) |
| `cls` | número de clase |

Son tensores de PyTorch; con `.cpu().numpy()` los pasamos a NumPy.

In [ ]:
resultado = modelo_coco.predict(muestra_val[0], conf=0.25, device=DEVICE, verbose=False)[0]

tabla = pd.DataFrame(resultado.boxes.xyxy.cpu().numpy().round(1), columns=["x1", "y1", "x2", "y2"])
tabla["clase"] = [resultado.names[int(c)] for c in resultado.boxes.cls]
tabla["confianza"] = resultado.boxes.conf.cpu().numpy().round(3)
tabla

## 8. Entrenamiento: nuestro propio detector de placas

Entrenar con Ultralytics es una sola llamada a `modelo.train()`. Partimos de `yolo26n.pt` (pesos de COCO) y Ultralytics reemplaza la "cabeza" del modelo por una con **1 clase**.

Hiperparámetros principales:

| Parámetro | Valor | Significado |
|-----------|-------|-------------|
| `epochs` | 20 | Pasadas completas por el set de entrenamiento |
| `imgsz` | 640 | Tamaño al que se redimensionan las imágenes (640×640) |
| `batch` | 16 | Imágenes por iteración. Si te quedas sin memoria, bájalo a 8 |
| `patience` | 10 | *Early stopping*: si el mAP en validación no mejora en 10 épocas, se detiene |
| `seed` | 42 | Semilla para reproducibilidad |

Detrás de esta llamada ocurre lo mismo que programamos a mano en el workshop: forward, cálculo de la loss, backward y actualización de pesos, época tras época. Ultralytics además aplica **aumentación de datos** automáticamente (mosaicos, volteos, cambios de color y de escala) y **evalúa en validación** al final de cada época.

**Tiempo estimado:** unos 50 segundos por época en una Mac con chip M3 (unos 17 minutos en total), menos en una GPU NVIDIA y bastante más en CPU. Para una prueba rápida, baja `EPOCHS` a 3.

Para no repetir el entrenamiento cada vez que abras el notebook: si ya existe un modelo entrenado, la celda lo reutiliza. Pon `ENTRENAR_DE_NUEVO = True` para forzar un nuevo entrenamiento.

In [ ]:
EPOCHS = 20
BATCH = 32
IMGSZ = 640
NOMBRE_RUN = "placas_yolo26n"
ENTRENAR_DE_NUEVO = True

SAVE_DIR = RUNS_DIR / NOMBRE_RUN          # aquí se guardan pesos, gráficos y métricas
BEST_PT = SAVE_DIR / "weights" / "best.pt"

if BEST_PT.exists() and not ENTRENAR_DE_NUEVO:
    print(f"Ya existe un modelo entrenado: {BEST_PT}")
else:
    modelo = YOLO("yolo26n.pt")           # pesos de COCO → transfer learning
    inicio = time.time()
    modelo.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        patience=10,
        device=DEVICE,
        seed=SEED,
        project=str(RUNS_DIR),
        name=NOMBRE_RUN,
        exist_ok=True,                    # sobrescribe la carpeta si vuelves a entrenar
        plots=True,
    )
    print(f"\nEntrenamiento completado en {(time.time() - inicio) / 60:.1f} minutos")

print("Archivos generados:")
for archivo in sorted(SAVE_DIR.iterdir()):
    print("  ", archivo.name)

### ¿Qué se guardó?

Ultralytics deja en `SAVE_DIR` todo lo necesario para analizar el entrenamiento:

- `weights/best.pt`: los pesos de la **mejor época** según el mAP en validación. **Este es el modelo que usaremos.**
- `weights/last.pt`: los pesos de la **última época** (sirve para reanudar un entrenamiento interrumpido).
- `results.csv`: las pérdidas y métricas de cada época.
- Imágenes: curvas, matriz de confusión, ejemplos de lotes de entrenamiento, etc.

> **Nota:** durante la lectura del dataset, Ultralytics puede avisar que encontró un **JPEG corrupto** y que lo "restauró". Lo repara y lo vuelve a guardar en su lugar; no afecta al entrenamiento. También crea archivos `.cache` en `labels/` para cargar las etiquetas más rápido la próxima vez.

Veamos cómo "ve" el modelo los datos durante el entrenamiento:

In [ ]:
mostrar(SAVE_DIR / "train_batch0.jpg", "Un lote de entrenamiento con aumentación de datos", figsize=(10, 10))
plt.show()

Esta imagen muestra la **aumentación mosaic**: Ultralytics pega **4 imágenes** en una sola, con recortes, escalas y colores distintos. Así el modelo ve placas en muchos contextos, tamaños y posiciones, y generaliza mejor con los mismos datos.

Un detalle: Ultralytics desactiva el mosaic en las últimas épocas (por defecto, las últimas 10) para que el modelo termine de ajustarse con imágenes "normales", parecidas a las que verá en la práctica.

## 9. Curvas de entrenamiento

El archivo `results.csv` guarda, para cada época, las **pérdidas** en train y validación y las **métricas** en validación.

YOLO26 optimiza varias pérdidas a la vez, porque debe aprender dos cosas distintas:

| Pérdida | ¿Qué mide? |
|---------|-----------|
| `box_loss` | Qué tan bien se ajusta la caja predicha a la caja real (**localización**) |
| `cls_loss` | Qué tan bien clasifica cada objeto (**clasificación**) |
| `l1_loss` | Error directo en las coordenadas de la caja (refinamiento) |

Cómo leer las curvas:

- Train y validación **bajan juntas** → el modelo sigue aprendiendo; podrías entrenar más épocas.
- Train baja pero validación **sube** → **sobreajuste** (*overfitting*): el modelo memoriza en lugar de generalizar.
- Ambas **estancadas y altas** → **subajuste**: el modelo es muy pequeño o necesita más épocas.

In [ ]:
df_res = pd.read_csv(SAVE_DIR / "results.csv")
df_res.columns = df_res.columns.str.strip()

perdidas = [c.removeprefix("train/") for c in df_res.columns if c.startswith("train/")]
metricas = ["metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]

fig, axes = plt.subplots(1, len(perdidas) + 1, figsize=(5 * (len(perdidas) + 1), 4))
for ax, perdida in zip(axes, perdidas):
    ax.plot(df_res["epoch"], df_res[f"train/{perdida}"], "o-", ms=3, label="train")
    ax.plot(df_res["epoch"], df_res[f"val/{perdida}"], "s-", ms=3, label="validación")
    ax.set(title=perdida, xlabel="época")
    ax.legend()
    ax.grid(alpha=0.3)

for metrica in metricas:
    axes[-1].plot(df_res["epoch"], df_res[metrica], "o-", ms=3,
                  label=metrica.split("/")[1].replace("(B)", ""))
axes[-1].set(title="Métricas en validación", xlabel="época", ylim=(0, 1))
axes[-1].legend()
axes[-1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

mejor = df_res.loc[df_res["metrics/mAP50-95(B)"].idxmax()]
print(f"Mejor época: {int(mejor['epoch'])} | "
      f"mAP50 = {mejor['metrics/mAP50(B)']:.3f} | mAP50-95 = {mejor['metrics/mAP50-95(B)']:.3f}")

### ¿Qué observamos?

- Las pérdidas de train y validación **bajan juntas**, sin señales de sobreajuste. Con más épocas probablemente seguiría mejorando un poco.
- Es normal que la pérdida de **validación** a veces sea **menor** que la de train: en train las imágenes vienen con aumentación (mosaicos, recortes, cambios de color), que las hace más difíciles.
- La `cls_loss` cae muy rápido: con una sola clase, decidir "esto es una placa" es fácil. Lo difícil es **ubicar la caja con precisión** (`box_loss`), que baja más lentamente.
- En las métricas de la derecha (las explicamos en la siguiente sección), mAP50, precisión y recall llegan a casi 1 desde las primeras épocas gracias al transfer learning. El **mAP50-95**, más exigente, es el que sigue mejorando época a época.

## 10. Evaluación: ¿qué tan bueno es el detector?

En clasificación bastaba con el **accuracy**: la predicción era correcta o incorrecta. En detección hay un problema nuevo: **¿cuándo una caja predicha "acierta"?** Casi nunca coincide al píxel con la caja real.

### IoU: Intersección sobre Unión

La medida que responde esa pregunta es el **IoU** (*Intersection over Union*):

$$\text{IoU} = \frac{\text{área de la intersección}}{\text{área de la unión}}$$

- IoU = 1: las cajas son idénticas.
- IoU = 0: las cajas no se tocan.
- Por convención, una predicción cuenta como **acierto** si su IoU con la caja real es **al menos 0.5**.

Programémoslo, porque lo usaremos en el análisis de errores:

In [ ]:
def iou(caja_a, caja_b):
    # IoU entre dos cajas en formato (x1, y1, x2, y2).
    # 1. Esquinas del rectángulo de intersección
    x1 = max(caja_a[0], caja_b[0])
    y1 = max(caja_a[1], caja_b[1])
    x2 = min(caja_a[2], caja_b[2])
    y2 = min(caja_a[3], caja_b[3])

    # 2. Áreas (si no se tocan, la intersección es 0)
    interseccion = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area_a = (caja_a[2] - caja_a[0]) * (caja_a[3] - caja_a[1])
    area_b = (caja_b[2] - caja_b[0]) * (caja_b[3] - caja_b[1])
    union = area_a + area_b - interseccion
    return interseccion / union if union > 0 else 0.0


# Casos de prueba
print(f"Cajas idénticas:           IoU = {iou([0, 0, 10, 10], [0, 0, 10, 10]):.2f}")
print(f"Desplazada a la mitad:     IoU = {iou([0, 0, 10, 10], [5, 0, 15, 10]):.2f}")
print(f"Una dentro de otra (36%):  IoU = {iou([0, 0, 10, 10], [2, 2, 8, 8]):.2f}")
print(f"Sin solapamiento:          IoU = {iou([0, 0, 10, 10], [20, 20, 30, 30]):.2f}")

Fíjate en el segundo caso: desplazar una caja **solo la mitad de su ancho** ya baja el IoU a 0.33, por debajo del umbral de 0.5. El IoU es exigente.

Veámoslo con el modelo entrenado sobre una imagen de validación: en verde la caja real, en rojo la predicha.

In [ ]:
modelo_best = YOLO(BEST_PT)


def cajas_reales(ruta_img):
    # Cajas ground truth de una imagen, en píxeles (x1, y1, x2, y2).
    ancho, alto = Image.open(ruta_img).size
    return yolo_a_xyxy(leer_etiquetas(ruta_etiqueta(ruta_img)), ancho, alto)


ruta = imgs_val[-20]
resultado = modelo_best.predict(ruta, conf=0.25, device=DEVICE, verbose=False)[0]
real = cajas_reales(ruta)[0]
predicha = resultado.boxes.xyxy.cpu().numpy()[0]

ax = mostrar(ruta, f"IoU entre la caja real y la predicha = {iou(real, predicha):.3f}", figsize=(9, 6))
dibujar_cajas(ax, [real], "lime", ["real"])
dibujar_cajas(ax, [predicha], "red", [f"predicha {resultado.boxes.conf[0]:.2f}"])
plt.show()

### Precisión, recall y mAP

Con el umbral de IoU podemos clasificar cada predicción:

- **Verdadero positivo (TP):** caja predicha con IoU ≥ 0.5 con una placa real.
- **Falso positivo (FP):** caja predicha que no corresponde a ninguna placa (o está mal ubicada).
- **Falso negativo (FN):** placa real que el modelo no detectó.

Y con eso, las métricas:

| Métrica | Fórmula | Pregunta que responde |
|---------|---------|----------------------|
| **Precisión** | TP / (TP + FP) | De las cajas que dibujé, ¿cuántas eran placas? |
| **Recall** | TP / (TP + FN) | De las placas que había, ¿cuántas encontré? |
| **mAP50** | Área bajo la curva precisión-recall, con IoU ≥ 0.5 | Resumen general del detector (0 a 1) |
| **mAP50-95** | Promedio del mAP con umbrales de IoU de 0.50 a 0.95 | Igual, pero exige cajas **muy bien ajustadas** |

El **mAP50-95** es la métrica principal en detección: premia no solo encontrar el objeto, sino ubicarlo con precisión.

Evaluamos el mejor modelo sobre todo el set de validación con `modelo.val()`:

In [ ]:
metricas_val = modelo_best.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    project=str(RUNS_DIR),
    name="placas_val",
    exist_ok=True,
    verbose=False,
)

print(f"Precisión: {metricas_val.box.mp:.3f}")
print(f"Recall:    {metricas_val.box.mr:.3f}")
print(f"mAP50:     {metricas_val.box.map50:.3f}")
print(f"mAP50-95:  {metricas_val.box.map:.3f}")

### ¿Cómo interpretar estos números?

- **Precisión, recall y mAP50 cercanos a 0.99:** el modelo encuentra prácticamente todas las placas y casi no inventa detecciones.
- **mAP50-95 ≈ 0.88:** más bajo, porque exige cajas muy bien ajustadas (IoU hasta 0.95). Encontrar la placa es fácil; dibujar su borde con precisión de pocos píxeles es lo difícil, sobre todo en placas pequeñas.
- Recuerda que la validación viene de unos pocos videos: son métricas **optimistas** respecto a lo que pasaría con fotos de otras ciudades, cámaras o condiciones de luz.

Ultralytics también guarda la **curva precisión-recall**. Cada punto de la curva corresponde a un umbral de confianza distinto: el **área bajo esta curva** es el mAP50.

In [ ]:
mostrar(Path(metricas_val.save_dir) / "BoxPR_curve.png", "Curva precisión-recall (validación)", figsize=(9, 6))
plt.show()

## 11. Predicciones del modelo entrenado

Las métricas resumen todo en un número. Ahora **miremos** las predicciones: en verde la caja real y en rojo la predicha con su confianza.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for ax, ruta in zip(axes.flat, random.sample(imgs_val, 8)):
    resultado = modelo_best.predict(ruta, conf=0.25, device=DEVICE, verbose=False)[0]
    predichas = resultado.boxes.xyxy.cpu().numpy()
    confianzas = resultado.boxes.conf.cpu().numpy()

    mostrar(ruta, f"{len(predichas)} placa(s) detectada(s)", ax=ax)
    dibujar_cajas(ax, cajas_reales(ruta), "lime")
    dibujar_cajas(ax, predichas, "red", [f"{c:.2f}" for c in confianzas])
plt.suptitle("Verde = caja real | Rojo = predicción del modelo (con su confianza)", fontsize=14)
plt.tight_layout()
plt.show()

En la mayoría de los casos la caja roja **coincide casi exactamente** con la verde. Lo notable: estas imágenes de validación **no se usaron para ajustar los pesos**, y el modelo aprendió a encontrar placas en solo 20 épocas, gracias a que partió de pesos preentrenados.

## 12. El umbral de confianza

Cada caja predicha trae una **confianza** entre 0 y 1. El parámetro `conf` decide a partir de qué valor aceptamos una detección:

- **Umbral bajo:** aceptamos casi todo → encontramos más placas (**recall alto**), pero aparecen cajas falsas (**precisión baja**).
- **Umbral alto:** solo aceptamos lo muy seguro → pocas cajas falsas (**precisión alta**), pero se escapan placas (**recall bajo**).

Midamos este equilibrio nosotros mismos. Hacemos una sola pasada con un umbral muy bajo (`conf=0.01`), guardamos todas las cajas y luego contamos TP, FP y FN para distintos umbrales.

In [ ]:
def contar_aciertos(cajas_pred, cajas_gt, umbral_iou=0.5):
    # Empareja predicciones con cajas reales (una a una) y devuelve (TP, FP, FN).
    # Las predicciones vienen ordenadas de mayor a menor confianza.
    emparejadas = set()
    tp = 0
    for caja in cajas_pred:
        candidatas = [(iou(caja, gt), g) for g, gt in enumerate(cajas_gt) if g not in emparejadas]
        if candidatas:
            mejor_iou, g = max(candidatas)
            if mejor_iou >= umbral_iou:
                emparejadas.add(g)
                tp += 1
    return tp, len(cajas_pred) - tp, len(cajas_gt) - tp


# Una sola pasada por validación con umbral muy bajo
predicciones = {}
for resultado in modelo_best.predict(imgs_val, conf=0.01, device=DEVICE, stream=True, verbose=False):
    orden = resultado.boxes.conf.cpu().numpy().argsort()[::-1]
    predicciones[resultado.path] = (resultado.boxes.xyxy.cpu().numpy()[orden],
                                    resultado.boxes.conf.cpu().numpy()[orden])

filas = []
for umbral in [0.05, 0.1, 0.25, 0.5, 0.75, 0.9]:
    tp = fp = fn = 0
    for ruta, (cajas, confs) in predicciones.items():
        t, f_p, f_n = contar_aciertos(cajas[confs >= umbral], cajas_reales(ruta))
        tp, fp, fn = tp + t, fp + f_p, fn + f_n
    filas.append({"umbral conf": umbral, "TP": tp, "FP": fp, "FN": fn,
                  "precisión": tp / max(tp + fp, 1), "recall": tp / max(tp + fn, 1)})

df_umbral = pd.DataFrame(filas).set_index("umbral conf")
df_umbral.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(df_umbral.index, df_umbral["precisión"], "o-", label="precisión")
ax.plot(df_umbral.index, df_umbral["recall"], "s-", label="recall")
ax.set(xlabel="umbral de confianza (conf)", ylabel="valor", ylim=(0, 1.05),
       title="Efecto del umbral de confianza (IoU ≥ 0.5)")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

### ¿Qué observamos?

- Al **subir** el umbral, los **falsos positivos bajan** (sube la precisión) pero los **falsos negativos suben** (baja el recall).
- Entre 0.1 y 0.75 el resultado casi no cambia: el modelo está **muy seguro** de sus detecciones correctas (confianzas altas) y los pocos errores tienen confianza más baja.
- Con un umbral de 0.9, en cambio, el recall cae fuerte: muchas placas reales se detectan con confianza entre 0.75 y 0.9 y quedan descartadas.
- El valor por defecto de Ultralytics, `conf=0.25`, suele ser un buen equilibrio.

El umbral adecuado depende de la aplicación:

- En un **peaje** que cobra automáticamente, no queremos perder ninguna placa → umbral **bajo** (priorizar recall) y que una persona revise los casos dudosos.
- En un sistema que **multa** automáticamente, una detección falsa es muy costosa → umbral **alto** (priorizar precisión).

## 13. Análisis de errores

Las métricas no dicen **dónde** falla el modelo. Para cada imagen de validación (con `conf=0.25`) calculamos:

- **Mejor IoU:** el IoU entre la placa real y la caja predicha que mejor la cubre. Si es menor a 0.5, el modelo no encontró la placa (falso negativo).
- **Falsos positivos:** cajas predichas que no corresponden a la placa anotada.

In [ ]:
filas = []
for ruta, (cajas, confs) in predicciones.items():
    cajas_025 = cajas[confs >= 0.5]
    tp, fp, fn = contar_aciertos(cajas_025, cajas_reales(ruta))
    mejor_iou = max((iou(cajas_reales(ruta)[0], c) for c in cajas_025), default=0.0)
    filas.append({"imagen": Path(ruta).name, "ruta": ruta, "n_predichas": len(cajas_025),
                  "mejor_iou": mejor_iou, "FP": fp, "FN": fn})

df_errores = pd.DataFrame(filas).sort_values("mejor_iou")

print(f"Placas encontradas (IoU ≥ 0.5): {(df_errores['FN'] == 0).sum()} de {len(df_errores)}")
print(f"Imágenes con falsos positivos:  {(df_errores['FP'] > 0).sum()}")
print(f"IoU promedio de las detecciones: {df_errores['mejor_iou'].mean():.3f}")

### Tipo 1: falsos positivos (cajas de más)

Veamos las imágenes donde el modelo dibujó cajas que no corresponden a la placa anotada:

In [ ]:
def mostrar_casos(filas_df, titulo):
    # Dibuja imágenes con la caja real (verde) y las predichas (rojo, con su confianza).
    fig, axes = plt.subplots(1, len(filas_df), figsize=(7 * len(filas_df), 5.5), squeeze=False)
    for ax, (_, fila) in zip(axes[0], filas_df.iterrows()):
        cajas, confs = predicciones[fila["ruta"]]
        mantener = confs >= 0.25
        mostrar(fila["ruta"], f"{fila['imagen']} | mejor IoU = {fila['mejor_iou']:.2f}", ax=ax)
        dibujar_cajas(ax, cajas_reales(fila["ruta"]), "lime")
        dibujar_cajas(ax, cajas[mantener], "red", [f"{c:.2f}" for c in confs[mantener]])
    plt.suptitle(titulo, fontsize=14)
    plt.tight_layout()
    plt.show()


con_fp = df_errores[df_errores["FP"] > 0]
if len(con_fp):
    mostrar_casos(con_fp.head(4), "Imágenes con falsos positivos (verde = real, rojo = predicción)")
else:
    print("No hay falsos positivos con conf = 0.25")

Mira con atención cada caja roja que no tiene una verde debajo. Con el modelo que entrenamos aparecen dos casos muy instructivos:

- **Una placa que nadie anotó:** en una de las imágenes hay un segundo vehículo al fondo con su placa visible. El modelo la detecta correctamente, pero como **no estaba anotada**, cuenta como falso positivo. Aquí el "error" es del **dataset**, no del modelo. Por eso siempre hay que revisar los errores a mano antes de sacar conclusiones de las métricas.
- **Texto que parece placa:** en otra imagen el modelo marca las letras del **nombre del modelo del auto** (por ejemplo "SAFARI") con confianza baja. Son letras grandes sobre un fondo liso, justo lo que el modelo aprendió a asociar con una placa. Es un falso positivo real; subir el umbral a 0.6 lo eliminaría.

### Tipo 2: cajas menos ajustadas

Ahora las imágenes con el **menor IoU**: el modelo encontró la placa, pero su caja es la que menos coincide con la anotada.

In [ ]:
mostrar_casos(df_errores.head(4), "Las 4 cajas menos ajustadas (verde = real, rojo = predicción)")

### ¿Qué observamos?

- Incluso los "peores" casos tienen IoU alrededor de 0.75 a 0.85: **todas las placas fueron encontradas**. Las diferencias son de pocos píxeles en el borde de la caja.
- Son, sobre todo, **placas pequeñas** (autos lejanos o en ángulo). En una placa de 30×10 píxeles, un error de 2 o 3 píxeles ya baja mucho el IoU. Esto explica por qué el mAP50-95 (≈ 0.88) es menor que el mAP50 (≈ 0.99).

Ideas para mejorar el modelo a partir de este análisis:

- **Corregir las anotaciones** (agregar las placas que faltan).
- Agregar ejemplos **negativos**: autos con textos o logos grandes sin placa visible.
- Usar un `imgsz` mayor (por ejemplo 960) o un modelo más grande (`yolo26s`) para ajustar mejor las placas pequeñas.
- Evaluar con fotos de **otras fuentes** (no solo estos videos) para medir la generalización real.

## 14. Aplicación: recortar las placas detectadas

Detectar la placa casi nunca es el objetivo final: normalmente queremos **leerla**. Un sistema de lectura de placas trabaja en dos etapas:

```
foto del auto → [detector YOLO] → recorte de la placa → [OCR] → "ABC-123"
```

El **OCR** (*Optical Character Recognition*) lee el texto, y funciona mucho mejor sobre un recorte limpio de la placa que sobre la foto completa. Nuestro detector resuelve la primera etapa.

Creamos una función que detecta y recorta las placas de una imagen:

In [ ]:
def recortar_placas(ruta_img, modelo=modelo_best, conf=0.25, margen=0.05):
    # Detecta placas y devuelve una lista de (recorte PIL, confianza).
    imagen = Image.open(ruta_img).convert("RGB")
    resultado = modelo.predict(imagen, conf=conf, device=DEVICE, verbose=False)[0]

    recortes = []
    for (x1, y1, x2, y2), confianza in zip(resultado.boxes.xyxy.cpu().numpy(),
                                           resultado.boxes.conf.cpu().numpy()):
        # Agrandamos la caja un poco (margen) para no cortar los bordes de la placa
        dx, dy = (x2 - x1) * margen, (y2 - y1) * margen
        caja = (max(0, x1 - dx), max(0, y1 - dy), min(imagen.width, x2 + dx), min(imagen.height, y2 + dy))
        recortes.append((imagen.crop(caja), float(confianza)))
    return recortes


fig, axes = plt.subplots(3, 4, figsize=(16, 6))
recortes = [r for ruta in random.sample(imgs_val, 20) for r in recortar_placas(ruta)][:12]
for ax, (recorte, confianza) in zip(axes.flat, recortes):
    mostrar(recorte, f"confianza {confianza:.2f}", ax=ax)
for ax in axes.flat[len(recortes):]:
    ax.axis("off")
plt.suptitle("Placas recortadas automáticamente por el detector", fontsize=14)
plt.tight_layout()
plt.show()

Notarás placas **repetidas** (la misma placa en varios recortes): son frames distintos del mismo video. Los recortes salen limpios y bien centrados, listos para la siguiente etapa.

Estos recortes serían la entrada de un OCR. Librerías como [EasyOCR](https://github.com/JaidedAI/EasyOCR) o [PaddleOCR](https://github.com/PaddlePaddle/PaddleOCR) pueden leer el texto de la placa en pocas líneas; lo dejamos como ejercicio.

## 15. Usar el modelo entrenado en una imagen nueva

El modelo entrenado es un archivo: `best.pt`. Para usarlo en otro notebook, script o aplicación solo necesitamos:

```python
from ultralytics import YOLO

modelo = YOLO("ruta/a/best.pt")
resultado = modelo.predict("foto.jpg", conf=0.25)[0]
```

Juntamos todo en una función que muestra la detección y los recortes lado a lado. Pruébala con tus propias fotos cambiando la ruta.

In [ ]:
def detectar_placas(ruta_img, modelo=modelo_best, conf=0.25):
    # Muestra la imagen con las placas detectadas y sus recortes.
    resultado = modelo.predict(ruta_img, conf=conf, device=DEVICE, verbose=False)[0]
    recortes = recortar_placas(ruta_img, modelo, conf)

    n_columnas = 1 + max(len(recortes), 1)
    fig, axes = plt.subplots(1, n_columnas, figsize=(6 + 3 * (n_columnas - 1), 4.5),
                             gridspec_kw={"width_ratios": [3] + [1] * (n_columnas - 1)})
    mostrar(resultado.plot()[..., ::-1], f"{len(recortes)} placa(s) detectada(s)", ax=axes[0])
    for ax, (recorte, confianza) in zip(axes[1:], recortes):
        mostrar(recorte, f"recorte ({confianza:.2f})", ax=ax)
    if not recortes:
        axes[1].axis("off")
    plt.tight_layout()
    plt.show()


# Cambia esta ruta por una foto tuya
detectar_placas(random.choice(imgs_val))

## 16. Cierre: ¿qué aprendimos?

1. **Detección = qué + dónde.** El modelo devuelve, por cada objeto, una caja, una clase y una confianza.
2. **Formato YOLO.** Una carpeta de imágenes, otra de etiquetas con el mismo nombre, y un `.txt` con `clase xc yc w h` normalizados. Un `data.yaml` le dice a Ultralytics dónde está todo.
3. **Transfer learning también funciona en detección.** Partiendo de pesos de COCO, un modelo nano aprendió a detectar placas con ~1,500 imágenes y 20 épocas.
4. **IoU es la base de la evaluación.** Decide si una caja acierta; con eso se calculan precisión, recall y mAP.
5. **El umbral de confianza es una decisión de negocio**: equilibra falsos positivos y falsos negativos según el costo de cada error.
6. **Mirar los errores** enseña más que las métricas globales, y muestra cómo mejorar el modelo.

### Ejercicios propuestos

1. **Modelo más grande:** entrena `yolo26s.pt` en lugar de `yolo26n.pt` (cambia también `NOMBRE_RUN`). Compara mAP50-95 y tiempo de entrenamiento. ¿Vale la pena?
2. **Resolución:** entrena con `imgsz=960`. ¿Mejora la detección de las placas pequeñas que aparecían en el análisis de errores?
3. **Sin transfer learning:** entrena desde cero con `YOLO("yolo26n.yaml")` (arquitectura sin pesos) las mismas 20 épocas. ¿Cuánto baja el mAP?
4. **Lectura de placas:** instala EasyOCR (`uv add easyocr`) y aplica `easyocr.Reader(["en"]).readtext()` a los recortes de la sección 14.
5. **Exportar:** convierte el modelo a ONNX con `modelo_best.export(format="onnx")`, un formato que se puede usar sin PyTorch en aplicaciones y servidores.

In [ ]:

import cv2 as cv

from ultralytics import YOLO

modelo = YOLO("/Users/marcell/Documents/codigo_by_tecsup/datascience-g8/notebooks/runs/placas_yolo26n/weights/best.pt")

cap = cv.VideoCapture("/Users/marcell/Documents/codigo_by_tecsup/datascience-g8/data/videoplayback.mp4")

while True:
    # Capture frame-by-frame
    ret, frame = cap.read()

    resultado = modelo.predict(frame, conf=0.25, device=DEVICE, verbose=False)[0]
    if len(resultado.boxes.xywhn):
        # print(resultado.boxes.xywhn)
        for coor in resultado.boxes.xyxy:
            x, y, w, h = coor.cpu().numpy()

            pt1 = (int(x), int(y))
            pt2 = (int(w), int(h))

            frame = cv.rectangle(frame, pt1, pt2, color=(255,0,0), thickness=3)

    # if frame is read correctly ret is True
    if not ret:
        print("Can't receive frame (stream end?). Exiting ...")
        break

    cv.imshow('frame', frame)
    if cv.waitKey(1) == ord('q'):
        break

# When everything done, release the capture
cap.release()
cv.destroyAllWindows()